# Initial imports (install if missing)

In [1]:
# ---- run once ----
import numpy as np
from numpy import nan
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import pandas as pd
from tabulate import tabulate
from matplotlib import cm
from datetime import datetime

## FUNCTION DEFINITIONS

In [2]:
def stim_100hz(dt_100hz=0.001, tf_100hz=100, amplitude_100hz=1):
    """Generate 100Hz stimulus pattern."""
    t_100hz = np.arange(0, tf_100hz, dt_100hz)
    freq_100hz = 100  # Frequency in Hz
    period_100hz = 1 / freq_100hz * 1000  # Period in ms
    pulse_width_100hz = 0.03  # Pulse width in ms
    I_100hz = np.zeros_like(t_100hz)

    for start_100hz in np.arange(0, tf_100hz, period_100hz):
        end_100hz = start_100hz + pulse_width_100hz
        I_100hz[(t_100hz >= start_100hz) & (t_100hz < end_100hz)] = amplitude_100hz
        
    intervals = int(tf_100hz / period_100hz)
    
    return t_100hz, I_100hz, intervals


def stim_latency(dt_latency=1, tf_latency=10, amplitude_latency=1):
    """Generate latency stimulus pattern."""
    t_latency = np.arange(0, tf_latency, dt_latency)
    freq_latency = 0.1  # Frequency in Hz
    period_latency = 1 / freq_latency * 10  # Period in ms
    pulse_width_latency = 0.1  # Pulse width in ms
    I_latency = np.zeros_like(t_latency)

    for start_latency in np.arange(0.1, tf_latency, period_latency):
        end_latency = start_latency + pulse_width_latency
        I_latency[(t_latency >= start_latency) & (t_latency < end_latency)] = amplitude_latency
    return t_latency, I_latency


def sigmoidal_ggap(sigmoid_percentage_values, max_L=135, k=0.129, x0=41, min_L=34.5):
    """Calculate gap junction conductance using sigmoidal function."""
    if sigmoid_percentage_values < 10.2:
        ggap_base = min_L + (max_L - min_L) / (1 + np.exp(-k * (sigmoid_percentage_values - x0)))
    else:
        ggap_base = max_L
    return ggap_base


def ninf(V): 
    """Potassium activation steady state."""
    return 1./(1 + np.exp((-53. - V) / 15.))


def minf_nat(V): 
    """Sodium activation steady state (NaT)."""
    return 1/(1+np.exp((V-v1_2m)/km))


def hinf_nat(V): 
    """Sodium inactivation steady state (NaT)."""
    return 1/(1+np.exp((V-v1_2h)/kh))


# Global parameters for NaT channel
v1_2m, km = -29.13, -8.92
v1_2h, kh = -47, 5


def initialize_neuron(compartments, initial_v, N, stimulus_mode, ninf, minf_nat, hinf_nat, first_interval=True):
    """Initialize neuron state variables."""
    v = np.zeros((compartments, N))
    n = np.zeros((compartments, N))
    m = np.zeros((compartments, N))
    h = np.zeros((compartments, N))
    s = np.zeros(N)

    if stimulus_mode == '100hz':
        initial_v_to_use = -68 if first_interval else -70
        for comp in range(compartments):
            v[comp, 0] = initial_v_to_use
            n[comp, 0] = ninf(initial_v_to_use)  
            m[comp, 0] = minf_nat(initial_v_to_use)
            h[comp, 0] = hinf_nat(initial_v_to_use)
    else:
        for comp in range(compartments):
            v[comp, 0] = initial_v
            n[comp, 0] = ninf(initial_v) 
            m[comp, 0] = minf_nat(initial_v)
            h[comp, 0] = hinf_nat(initial_v)

    return v, n, m, h, s


def get_stimulus(t, dt, mode, amplitude):
    """Get stimulus pattern based on mode."""
    if mode == 'latency':
        t_latency, I_latency = stim_latency(tf_latency=t, dt_latency=dt, amplitude_latency=amplitude)
        return t_latency, I_latency, None
    elif mode == '100hz':
        t_100hz, I_100hz, intervals = stim_100hz(tf_100hz=t, dt_100hz=dt, amplitude_100hz=amplitude)
        return t_100hz, I_100hz, intervals
    else:
        raise ValueError("Invalid stimulus mode selected. Choose either 'latency' or '100hz'.")


def myneuron(gL, gk, gna, gel, gsyn, model_percentage_filled, noise_level, seed,
             max_L, k, x0, min_L, taud, taur, Vsyn, stimulus_mode, amplitude=1):
    """Main neuron simulation function."""
    
    # Set time parameters based on stimulus mode
    if stimulus_mode == 'latency':
        dt = 0.0001  # Time step for latency (ms)
        t = 10  # Total time for latency (ms)
    elif stimulus_mode == '100hz':
        dt = 0.001  # Time step for 100hz (ms)
        t = 100  # Total time for 100hz (ms)
    else:
        raise ValueError("Invalid stimulus mode selected. Choose either 'latency' or '100hz'.")

    N = round(t/dt)

    if seed is None:
        seed = np.random.randint(0, 10000)
    np.random.seed(seed)
    
    # Define local functions
    def ninf(V): return 1./(1 + np.exp((-53. - V) / 15.)*1)
    def taun(V): return taun_multiplier * (1.1 + 4.7 * np.exp(-((-79. - V) / 50.)**2))
    def minf(V): return 1./(1 + np.exp((-40. - V) / 15.))
    def taum(V): return 0.04 + 0.46 * np.exp(-((-38. - V) / 30.)**2)
    def hinf(V): return 1./(1. + np.exp((-62. - V) / -7.))
    def tauh(V): return 1.2 + 7.4 * np.exp(-((-67. - V) / 20.)**2)
    
    def minf_nat(V): return 1/(1+np.exp((V-v1_2m)/km))
    def mtau_nat(V): return 0.13 + 3.43/(1+np.exp((V+45.35)/5.98))
    def hinf_nat(V): return 1/(1+np.exp((V-v1_2h)/kh))
    def htau_nat(V): return 0.36 + np.exp((V+20.65)/-10.47)
    
    # Reversal potentials
    El = -85   # Leak reversal potential (mV)
    Ek = -74   # Potassium reversal potential (mV)
    Ena = 65   # Sodium reversal potential (mV)
    
    # Initialize neuron
    v, n, m, h, s = initialize_neuron(4, -70 if stimulus_mode == 'latency' else -68, N, 
                                      stimulus_mode, ninf, minf_nat, hinf_nat)

    ggap_values = np.zeros(N)
    
    # Generate the stimulus
    time, stimulus, intervals = get_stimulus(t=t, dt=dt, mode=stimulus_mode, amplitude=amplitude)

    # Main simulation loop
    for i in range(N-1):
        ggap_base_unfinished = sigmoidal_ggap(model_percentage_filled, max_L, k, x0, min_L)
        ggap_base = ggap_base_unfinished / (1 + np.exp(-k * (model_percentage_filled - x0)))
        
        ggap_random_factor = np.random.uniform(1 - noise_level, 1 + noise_level)
        ggap = ggap_base * ggap_random_factor
        
        ggap_values[i] = ggap
        
        taun_multiplier = 1
        taur = 0.1  
        taud = 3
        gsyn = 0.08 
        Vsyn = 0
        Istim = stimulus[i]
        
        # Compartment 0 (stimulated)
        v[0, i+1] = v[0, i] + dt*(-gna*h[0, i]*(m[0, i]**3)*(v[0, i]-Ena) - gk*(n[0, i]**4)*(v[0, i]-Ek) - gL*(v[0, i]-El) + Istim - gel*(v[0, i]-v[1, i]))
        n[0, i+1] = n[0, i] + dt*((ninf(v[0, i]) - n[0, i])/taun(v[0, i]))
        m[0, i+1] = m[0, i] + dt*((minf_nat(v[0, i]) - m[0, i])/mtau_nat(v[0, i]))
        h[0, i+1] = h[0, i] + dt*((hinf_nat(v[0, i]) - h[0, i])/htau_nat(v[0, i]))
        
        # Compartment 1 (intermediate)
        v[1, i+1] = v[1, i] + dt*(-gna*h[1, i]*(m[1, i]**3)*(v[1, i]-Ena) - gk*(n[1, i]**4)*(v[1, i]-Ek) - gL*(v[1, i]-El) - gel*(v[1, i]-v[0, i]) - gel*(v[1, i]-v[2, i]))
        n[1, i+1] = n[1, i] + dt*((ninf(v[1, i]) - n[1, i])/taun(v[1, i]))
        m[1, i+1] = m[1, i] + dt*((minf_nat(v[1, i]) - m[1, i])/mtau_nat(v[1, i]))
        h[1, i+1] = h[1, i] + dt*((hinf_nat(v[1, i]) - h[1, i])/htau_nat(v[1, i]))
        
        # Compartment 2 (gap junction source)
        v[2, i+1] = v[2, i] + dt*(-gna*h[2, i]*(m[2, i]**3)*(v[2, i]-Ena) - gk*(n[2, i]**4)*(v[2, i]-Ek) - gL*(v[2, i]-El) - gel*(v[2, i]-v[1, i]) - ggap*(v[2, i]-v[3, i]))
        n[2, i+1] = n[2, i] + dt*((ninf(v[2, i]) - n[2, i])/taun(v[2, i]))
        m[2, i+1] = m[2, i] + dt*((minf_nat(v[2, i]) - m[2, i])/mtau_nat(v[2, i]))
        h[2, i+1] = h[2, i] + dt*((hinf_nat(v[2, i]) - h[2, i])/htau_nat(v[2, i]))
        
        # Synaptic variable
        s[i+1] = s[i] + dt * ((1 + np.tanh(v[2,i]/4))/2 * (1-s[i])/taur - s[i]/taud)
        Isyn = gsyn*s[i]*(Vsyn - v[3,i])
        Isyn_values = np.zeros(N)
        Isyn_values[i] = Isyn  
        
        # Compartment 3 (postsynaptic)
        v[3, i+1] = v[3, i] + dt*(-gna*h[3, i]*(m[3, i]**3)*(v[3, i]-Ena) - gk*(n[3, i]**4)*(v[3, i]-Ek) - gL*(v[3, i]-El) - ggap*(v[3, i]-v[2 ,i]) - Isyn)
        n[3, i+1] = n[3, i] + dt*((ninf(v[3, i]) - n[3, i])/taun(v[3, i]))
        m[3, i+1] = m[3, i] + dt*((minf_nat(v[3, i]) - m[3, i])/mtau_nat(v[3, i]))
        h[3, i+1] = h[3, i] + dt*((hinf_nat(v[3, i]) - h[3, i])/htau_nat(v[3, i]))

    return time, stimulus, v, n, m, h, s, seed


def find_highest_peaks_in_intervals(v, time, intervals, threshold=30):
    """Find highest peaks in each interval for each compartment."""
    peaks_dict = {i: [] for i in range(v.shape[0])}  
    interval_length = len(time) // intervals  
    
    for comp in range(v.shape[0]):  
        for interval_index in range(intervals):  
            start_idx = interval_index * interval_length
            end_idx = start_idx + interval_length

            v_interval = v[comp, start_idx:end_idx]
            time_interval = time[start_idx:end_idx]

            peak_indices, properties = find_peaks(v_interval, height=threshold)

            if peak_indices.size > 0:                  
                max_peak_idx = peak_indices[np.argmax(properties['peak_heights'])]
                peak_idx = start_idx + max_peak_idx  
                peak_voltage = v_interval[max_peak_idx]
                peaks_dict[comp].append((interval_index, peak_idx, peak_voltage))
            else:
                peaks_dict[comp].append((interval_index, None, None))

    return peaks_dict


def calculate_time_delays(peaks_dict, intervals, dt):    
    """Calculate time delays between compartments."""
    delay_dict = {interval: [] for interval in range(intervals)}  
    
    for interval in range(intervals):
        peak_indices = []
        for comp in peaks_dict:
            interval_peaks = [peak for peak in peaks_dict[comp] if peak[0] == interval]
            if interval_peaks and interval_peaks[0][1] is not None:
                peak_indices.append(interval_peaks[0][1])
            else:
                peak_indices.append(None)  
        for i in range(len(peak_indices) - 1):
            if peak_indices[i] is not None and peak_indices[i+1] is not None:
                delay = (peak_indices[i+1] - peak_indices[i]) * dt  
                delay_dict[interval].append(delay)
            else:
                delay_dict[interval].append(None)  
    
    return delay_dict


def pad_list_to_length(lst, length, pad_value=None):
    """Pad list to specified length."""
    return lst + [pad_value] * (length - len(lst))


def first_crossing_idx(trace, thresh):
    """Index of the first sample where `trace` crosses `thresh` upward."""
    above = trace >= thresh
    if not above.any():
        return None
    return np.argmax(above)


def latency_ms(time, v, pre_comp=0, post_comp=3, thresh=60):
    """
    Calculate latency (ms) between first threshold crossing of `pre_comp`
    and `post_comp`. Returns (latency, idx_pre, idx_post).
    """
    idx_pre = first_crossing_idx(v[pre_comp], thresh)
    idx_post = first_crossing_idx(v[post_comp], thresh)
    if idx_pre is None or idx_post is None:
        return None, idx_pre, idx_post
    return (time[idx_post] - time[idx_pre]), idx_pre, idx_post


# ───────────────────────── PLOTTING HELPERS ─────────────────────────

def plot_stimulus(time, stimulus, title_suffix, savepath=None):
    plt.figure(figsize=(10, 4))
    plt.plot(time, stimulus, color='magenta', label='Stimulus')
    plt.title(f'{title_suffix} Stimulus')
    plt.xlabel('Time (ms)')
    plt.ylabel('Stimulus Intensity')
    plt.grid(True)
    plt.legend()
    if savepath:
        plt.savefig(savepath, bbox_inches="tight")
    plt.show()


def plot_100hz_traces(time, v, model_percentage_filled):
    plt.figure(figsize=(10, 8))
    plt.plot(time, v[0, :], label='Compartment 1 Voltage')
    plt.plot(time, v[3, :], label='Compartment 4 Voltage')
    for x in np.arange(0, max(time)+1, 10):
        plt.axvline(x, color='red', linestyle='--', linewidth=0.5,
                    label='Stimulus Onset' if x == 0 else None)
    plt.title(f'Membrane Potential Over Time at {model_percentage_filled}%')
    plt.xlabel('Time (ms)')
    plt.ylabel('Membrane Potential (mV)')
    plt.legend(loc="upper right")
    plt.show()


def plot_100hz_response_curve(results_df, runs_per_trial):
    df = results_df.copy()
    df['model_percentage_filled'] = df['model_percentage_filled'].apply(lambda x: f"{x}%")
    df['compartment_4_peak_detected'] = df['compartment_4_peak_detected'].astype(int)

    percentage_values = df['model_percentage_filled'].unique()
    intervals = df['interval'].max()

    plt.figure(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0, 1, len(percentage_values)))
    for i, percentage in enumerate(percentage_values):
        filtered = df[df['model_percentage_filled'] == percentage]
        interval_means = filtered.groupby('interval')['compartment_4_peak_detected'].sum()
        plt.plot(interval_means.index, interval_means.values,
                 marker='o', linestyle='-', color=colors[i],
                 label=f'{percentage} Filled')

    plt.xlabel('Interval Number')
    plt.ylabel('Firing Probability (%)')
    plt.title('Average Firing Probability per Interval Across Runs n=' + str(runs_per_trial) + ' runs')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.xticks(range(1, intervals + 1))
    plt.show()


def plot_latency_full(time, v, idx0, idx3, latency, savepath=None):
    plt.figure(figsize=(10, 8))
    plt.plot(time, v[0, :], label='Compartment 1 Voltage')
    plt.plot(time, v[1, :], label='Compartment 2 Voltage')
    plt.plot(time, v[2, :], label='Compartment 3 Voltage')
    plt.plot(time, v[3, :], label='Compartment 4 Voltage')

    if idx0 is not None:
        plt.axvline(time[idx0], color='red', linestyle='--', linewidth=1, label='C0 60 mV')
    if idx3 is not None:
        plt.axvline(time[idx3], color='green', linestyle='--', linewidth=1, label='C3 60 mV')

    if idx0 is not None and idx3 is not None:
        mid_t = (time[idx0] + time[idx3]) / 2
        ymax  = max(v[0, idx0], v[3, idx3]) + 5
        plt.annotate(f'{latency:.2f} ms',
                     xy=(mid_t, ymax), xytext=(0, 8), textcoords='offset points',
                     ha='center', va='bottom',
                     arrowprops={'arrowstyle':'<->'})
    plt.ylim(-72, 85)
    plt.title('Membrane Potential Over Time')
    plt.xlabel('Time (ms)')
    plt.ylabel('Membrane Potential (mV)')
    plt.legend(loc="upper right")
    if savepath:
        plt.savefig(savepath, bbox_inches="tight")
    plt.show()


def plot_latency_panel(time_pre,
                       v_pre,
                       post_dict,     # {'10 %': v10[3], '5 %': v5[3], ...}
                       thresh=60,
                       ylims=(-72, 80),
                       savepath=None):
    """Small panel like 2nd image."""
    def _first_crossing_idx(trace, thr):
        above = trace >= thr
        return int(np.argmax(above)) if above.any() else None

    latencies = {}
    color_cycle = ['red', 'orange', 'tab:blue', 'tab:green', 'tab:purple', 'tab:brown']

    fig, ax = plt.subplots(figsize=(6.2, 4.6), dpi=130)

    # Pre trace
    ax.plot(time_pre, v_pre, color='black', lw=1.5, label='Compartment 1 Voltage')
    idx_pre = _first_crossing_idx(v_pre, thresh)
    if idx_pre is not None:
        ax.axvline(time_pre[idx_pre], color='black', ls='--', lw=0.8)

    # Post traces
    for i, (lbl, trace) in enumerate(post_dict.items()):
        col = color_cycle[i % len(color_cycle)]
        ax.plot(time_pre, trace, color=col, lw=1.8, label=f'Compartment 4 Voltage at {lbl}')
        idx_post = _first_crossing_idx(trace, thresh)

        if idx_pre is not None and idx_post is not None:
            ax.axvline(time_pre[idx_post], color=col, ls='--', lw=0.8)
            t0, t1 = time_pre[idx_pre], time_pre[idx_post]
            mid_t  = (t0 + t1) / 2
            ymax   = max(v_pre[idx_pre], trace[idx_post]) + 5 + 4*i
            ax.annotate(f'{(t1 - t0):.2f} ms',
                        xy=(t0, ymax), xytext=(t1, ymax),
                        ha='center', va='bottom', fontsize=8, color=col,
                        arrowprops=dict(arrowstyle='<->', lw=0.8, color=col))
            latencies[lbl] = (t1 - t0)
        else:
            latencies[lbl] = None

    ax.axhline(thresh, color='gray', ls=':', lw=0.6)
    ax.set_ylim(*ylims)
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Membrane Potential (mV)')
    ax.set_title('Membrane Potential Over Time')
    ax.legend(loc='upper right', frameon=False, fontsize=8)
    if savepath:
        plt.savefig(savepath, bbox_inches='tight')
    plt.show()

    return latencies



def run_100hz(amplitude_100hz, gL, gk, gna, gel, gsyn,
              mpf_list, noise_level, seed,
              max_L, k, x0, min_L, taud, taur, Vsyn,
              trials, runs_per_trial):
    dt, t = 0.001, 100
    N = round(t/dt)
    results_df = pd.DataFrame()
    c4_traces = {}          # <── store C4 for each % filled
    c0_trace  = None        # keep one C0 (they’re identical across mpf)

    for trial in range(trials):
        trial_seed = seed + trial if seed is not None else np.random.randint(0, 10000)
        np.random.seed(trial_seed)
        run_seed = trial_seed

        for run in range(runs_per_trial):
            np.random.seed(run_seed)
            first_interval = (run == 0)
            v, n, m, h, s = initialize_neuron(4, -70, N, '100hz', ninf, minf_nat, hinf_nat, first_interval)

            for mpf in mpf_list:
                time, stimulus, v, n, m, h, s, used_seed = myneuron(
                    gL=gL, gk=gk, gna=gna, gel=gel, gsyn=gsyn,
                    model_percentage_filled=mpf,
                    noise_level=noise_level, seed=run_seed,
                    max_L=max_L, k=k, x0=x0, min_L=min_L,
                    taud=taud, taur=taur, Vsyn=Vsyn,
                    stimulus_mode='100hz', amplitude=amplitude_100hz
                )
                run_seed += 1

                if c0_trace is None:
                    c0_trace = v[0].copy()
                c4_traces[mpf] = v[3].copy()

                _, _, intervals = get_stimulus(t=len(time)*dt, dt=dt,
                                               mode='100hz', amplitude=amplitude_100hz)
                peaks_dict = find_highest_peaks_in_intervals(v, time, intervals, threshold=30)
                delay_dict = calculate_time_delays(peaks_dict, intervals, dt)
                comp4_delays = [delay_dict[i][-1] for i in range(intervals)]

                rows = []
                for interval in range(intervals):
                    peak_voltage  = peaks_dict[3][interval][2]
                    peak_detected = peak_voltage is not None
                    delay         = comp4_delays[interval]
                    rows.append({
                        'interval': interval,
                        'model_percentage_filled': mpf,
                        'compartment_4_peak_detected': peak_detected,
                        'compartment_4_peak_voltage': peak_voltage,
                        'comp1_to_comp4_delay': delay
                    })
                results_df = pd.concat([results_df, pd.DataFrame(rows)], ignore_index=True)

    return {'time': time, 'stimulus': stimulus, 'c0': c0_trace,
            'c4': c4_traces, 'results_df': results_df, 'dt': dt}


def plot_100hz_two_colors(time, c0, c4_red, c4_orange,
                          label_red, label_orange):
    plt.figure(figsize=(10, 6))
    plt.plot(time, c0, color='black',  lw=1.5, label='Compartment 0 Voltage')
    plt.plot(time, c4_red,    color='red',    lw=1.8, label=f'Compartment 4 ({label_red})')
    plt.plot(time, c4_orange, color='orange', lw=1.8, label=f'Compartment 4 ({label_orange})')
    for x in np.arange(0, max(time)+1, 10):
        plt.axvline(x, color='red', linestyle='--', linewidth=0.5,
                    label='Stimulus Onset' if x == 0 else None)
    plt.xlabel('Time (ms)')
    plt.ylabel('Membrane Potential (mV)')
    plt.title('Membrane Potential Over Time (100 Hz)')
    plt.legend(loc='upper right')
    plt.show()




def run_latency(amplitude, gL, gk, gna, gel, gsyn,
                model_percentage_filled_values, noise_level, seed,
                max_L, k, x0, min_L, taud, taur, Vsyn):
    dt, t = 0.0001, 10
    results_df = pd.DataFrame()

    # main condition (first value in list)
    mpf_main = model_percentage_filled_values[0]
    time, stimulus, v, n, m, h, s, used_seed = myneuron(
        gL, gk, gna, gel, gsyn, mpf_main, noise_level, seed,
        max_L, k, x0, min_L, taud, taur, Vsyn,
        stimulus_mode='latency', amplitude=amplitude
    )
    latency, idx0, idx3 = latency_ms(time, v, pre_comp=0, post_comp=3, thresh=60)

    data = {
        'model_percentage_filled': mpf_main,
        'time': time,
        'stimulus': stimulus,
        'compartment_1_voltage': v[0, :],
        'compartment_2_voltage': v[1, :],
        'compartment_3_voltage': v[2, :],
        'compartment_4_voltage': v[3, :],
        'seed': used_seed
    }
    results_df = pd.concat([results_df, pd.DataFrame(data)], ignore_index=True)

    # extra comparison run (hard-coded 5% like before; change if needed)
    time2, stim2, v2, *_ = myneuron(
        gL=gL, gk=gk, gna=gna, gel=gel, gsyn=gsyn,
        model_percentage_filled=5,
        noise_level=noise_level, seed=seed,
        max_L=max_L, k=k, x0=x0, min_L=min_L,
        taud=taud, taur=taur, Vsyn=Vsyn,
        stimulus_mode='latency', amplitude=amplitude
    )
    lat2, _, idx3_2 = latency_ms(time2, v2, pre_comp=0, post_comp=3, thresh=60)

    return {
        'time': time,
        'stimulus': stimulus,
        'v': v,
        'idx0': idx0,
        'idx3': idx3,
        'latency': latency,
        'extra_time': time2,
        'extra_v': v2,
        'latency2': lat2,
        'idx3_2': idx3_2,
        'results_df': results_df,
        'dt': dt
    }


# Run the simulation

In [ ]:
# =====================================================================================
# 100 Hz genotype plotting from Excel (sheet: "For Code Plots")
#  + combined overlay with your simulation 100 Hz curves in ONE figure
#  + parallelized 100 Hz sweep across mpf values
# =====================================================================================

EXCEL_PATH = r"/Users/juanlopez2016/Desktop/Lab work/Frazzled:DCC revision data/Data for Figures.xlsx"
SHEET_NAME = "For Code Plots"

# Mapping from your “nice” labels -> row labels in the Excel sheet
GENOTYPE_MAP = {
    "Control Sibling (9.04% GJ antibody)":      "w; fra3/CyO; R91H05:GFP/+",
    "Frazzled LOF (5.31% GJ antibody)":         "w; fra3/4; R91H05:GFP/+",
    "Full length Frazzled (8.67% GJ antibody)": "w; fra3/4; R91H05:GFP/UAS-Frazzled",
    "Frazzled E1354A (4.18% GJ antibody)":      "w; fra3/4; R91H05:GFP/UAS-FraE1354A",
    "Frazzled ICD (10.59% GJ antibody)":        "w; fra3/4; R91H05:GFP/UAS-FraICD",
    "ShakB2 LOF (0.10% GJ antibody)":           "ShakB2 Ablation",
}

TRIAL_COLS_100HZ = [f"100 Hz Genotype Total Trial {i}" for i in range(1, 11)]


def load_genotype_trials_100hz(
    excel_path: str = EXCEL_PATH,
    sheet_name: str = SHEET_NAME,
    genotype_map: dict = GENOTYPE_MAP,
):
    """
    Returns:
      data_dict: {pretty_label: np.array shape (10,) of totals per interval}
      df_sheet : full sheet dataframe (with 'Genotypes' column)
    """
    df = pd.read_excel(excel_path, sheet_name=sheet_name).copy()
    if "Genotypes" not in df.columns and "Unnamed: 0" in df.columns:
        df.rename(columns={"Unnamed: 0": "Genotypes"}, inplace=True)

    data_dict = {}
    for pretty_label, raw_label in genotype_map.items():
        row = df.loc[df["Genotypes"] == raw_label]
        if row.empty:
            continue
        y = row.iloc[0][TRIAL_COLS_100HZ].to_numpy(dtype=float)
        data_dict[pretty_label] = y

    return data_dict, df


def plot_100hz_genotypes_from_sheet(
    excel_path: str = EXCEL_PATH,
    sheet_name: str = SHEET_NAME,
    genotype_map: dict = GENOTYPE_MAP,
    ylims: tuple | None = None,
    savepath: str | None = "100hz_genotypes_from_sheet.pdf",
):
    """Plots the 10 interval totals for the requested genotypes from the Excel sheet."""
    data_dict, _ = load_genotype_trials_100hz(excel_path, sheet_name, genotype_map)
    x = np.arange(1, 11)

    fig, ax = plt.subplots(figsize=(9, 4))
    for label, y in data_dict.items():
        ax.plot(x, y, marker="o", linewidth=2, label=label)

    ax.set_title("100 Hz totals by genotype (spreadsheet)")
    ax.set_xlabel("Interval (1–10)")
    ax.set_ylabel("100 Hz Genotype Total")
    ax.set_xticks(x)
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(fontsize=8, ncol=2, loc="upper right")

    if ylims is not None:
        ax.set_ylim(*ylims)

    fig.tight_layout()
    if savepath:
        fig.savefig(savepath, dpi=300)
    return fig, ax


# ========================= NEW: combined overlay in ONE figure ========================

def _load_100hz_sheet_series(excel_path=EXCEL_PATH, sheet_name=SHEET_NAME, genotype_map=GENOTYPE_MAP):
    """Return dict {pretty_label: np.array of 10 interval totals} from the Excel sheet."""
    data_dict, _ = load_genotype_trials_100hz(excel_path, sheet_name, genotype_map)
    return data_dict

def _sim_interval_curve(results_df, mpf_value, runs_per_trial):
    """
    Build a 10-point (interval 1..10) response-frequency curve for one model % (mpf_value).
    Tries to be robust to different result_df schemas.
    """
    sub = results_df[results_df["mpf"] == mpf_value].copy()
    if "interval" in sub.columns:
        sub["interval"] = sub["interval"].astype(int)

    for col in ("response_freq", "response_frequency", "percent", "pct"):
        if col in sub.columns:
            s = sub.sort_values("interval")[col].to_numpy(dtype=float)
            if s.size == 10:
                return s

    if "fired" in sub.columns:
        s = (sub.groupby("interval")["fired"].mean() * 100.0)\
             .reindex(range(1, 11), fill_value=np.nan).to_numpy()
        return s

    for hits_col in ("successes", "hits", "n_hits", "count"):
        if hits_col in sub.columns:
            s = (sub.groupby("interval")[hits_col].mean() / runs_per_trial * 100.0)\
                 .reindex(range(1, 11), fill_value=np.nan).to_numpy()
            return s

    raise ValueError("results_df does not contain expected columns to compute response frequency.")

def plot_100hz_combined(
    out100,
    runs_per_trial: int,
    mpf_to_plot=(10, 8, 6, 4, 0),   # choose which model % lines to show
    excel_path=EXCEL_PATH,
    sheet_name=SHEET_NAME,
    savepath="100hz_sim_plus_genotypes.pdf",
):
    """
    Creates a single figure with:
      • Simulation curves: “{mpf}% Gap Junctions in compartment 3” (solid lines, circle markers)
      • Spreadsheet genotypes: the six specified (dashed lines, square markers)
    """
    x = np.arange(1, 11)

    geno_series = _load_100hz_sheet_series(excel_path, sheet_name, GENOTYPE_MAP)

    resdf = out100["results_df"].copy()
    if "interval" in resdf.columns:
        resdf["interval"] = resdf["interval"].astype(int)

    fig, ax = plt.subplots(figsize=(11, 6))

    # Simulation lines
    for mpf in mpf_to_plot:
        if (resdf["mpf"] == mpf).any():
            y = _sim_interval_curve(resdf, mpf, runs_per_trial)
            ax.plot(x, y, marker="o", linewidth=2.5, label=f"{mpf}% Gap Junctions in compartment 3")

    # Spreadsheet genotype lines
    for label, y in geno_series.items():
        ax.plot(x, y, linestyle="--", marker="s", linewidth=2.5, label=label)

    ax.set_xlim(1, 10)
    ax.set_xticks(x)
    ax.set_ylim(0, 100)
    ax.set_xlabel("Stimulus Interval", fontsize=12)
    ax.set_ylabel("Response Frequency (%)", fontsize=12)
    ax.grid(True, axis="y", alpha=0.25)
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=9, framealpha=1)
    fig.tight_layout()

    if savepath:
        fig.savefig(savepath, dpi=300)
    return fig, ax


# ===================== Parallel sweep across % filled (mpf) — joblib version =====================
# Avoids BrokenProcessPool by using joblib's 'loky' backend (cloudpickle).
# Falls back to threads or pure-serial if needed.
#
# Assumes: pandas as pd is already imported elsewhere in your file.

import os
import traceback

def _run_100hz_one_mpf_job(
    mpf, amp_100hz, gL, gk, gna, gel, gsyn,
    noise_level, seed, max_L, k, x0, min_L, taud, taur, Vsyn,
    trials, runs_per_trial
):
    """Run your existing run_100hz for a single mpf and return (mpf, out)."""
    out = run_100hz(
        amp_100hz, gL, gk, gna, gel, gsyn,
        [mpf], noise_level, seed,
        max_L, k, x0, min_L, taud, taur, Vsyn,
        trials, runs_per_trial
    )
    return mpf, out

def _merge_run_100hz_parts(hz_mpf_list, parts):
    """Merge per-mpf outputs to match the shape of run_100hz over a list."""
    parts.sort(key=lambda t: hz_mpf_list.index(t[0]))  # preserve requested order
    merged = {"time": None, "c0": None, "c4": {}, "results_df": None}
    for mpf, out in parts:
        if merged["time"] is None: merged["time"] = out.get("time")
        if merged["c0"]   is None: merged["c0"]   = out.get("c0")

        c4_block = out.get("c4", {})
        if isinstance(c4_block, dict):
            merged["c4"].update(c4_block)
        else:
            merged["c4"][mpf] = c4_block

        df = out.get("results_df")
        if df is not None:
            merged["results_df"] = df if merged["results_df"] is None else pd.concat(
                [merged["results_df"], df], ignore_index=True
            )

    if merged["results_df"] is not None and "interval" in merged["results_df"].columns:
        merged["results_df"]["interval"] = merged["results_df"]["interval"].astype(int)
        merged["results_df"] = merged["results_df"].sort_values(["mpf", "interval"]).reset_index(drop=True)
    return merged

def run_100hz_parallel_over_mpf(
    amp_100hz, gL, gk, gna, gel, gsyn,
    hz_mpf_list, noise_level, seed,
    max_L, k, x0, min_L, taud, taur, Vsyn,
    trials, runs_per_trial,
    n_jobs: int | None = None
):
    """
    Parallelizes across entries in hz_mpf_list (one process per mpf) using joblib's 'loky' backend.
    Returns a merged output dict with the same structure as run_100hz(hz_mpf_list=...).

    If joblib is unavailable or a worker errors, falls back to ThreadPool, then to serial.
    """
    if n_jobs is None:
        n_jobs = max(1, (os.cpu_count() or 2) - 1)
    n_jobs = min(n_jobs, len(hz_mpf_list))

    # ---- Try joblib (preferred) ----
    try:
        from joblib import Parallel, delayed
        parts = Parallel(n_jobs=n_jobs, backend="loky", prefer="processes")(
            delayed(_run_100hz_one_mpf_job)(
                mpf, amp_100hz, gL, gk, gna, gel, gsyn,
                noise_level, seed, max_L, k, x0, min_L, taud, taur, Vsyn,
                trials, runs_per_trial
            )
            for mpf in hz_mpf_list
        )
        return _merge_run_100hz_parts(hz_mpf_list, parts)

    except Exception as e:
        print("[parallel warning] joblib 'loky' failed; falling back to ThreadPool.")
        print("Reason:", repr(e))
        print(traceback.format_exc())

    # ---- Fallback: threads (works even when processes can't start) ----
    try:
        from concurrent.futures import ThreadPoolExecutor, as_completed
        parts = []
        with ThreadPoolExecutor(max_workers=n_jobs) as ex:
            futs = [
                ex.submit(
                    _run_100hz_one_mpf_job,
                    mpf, amp_100hz, gL, gk, gna, gel, gsyn,
                    noise_level, seed, max_L, k, x0, min_L, taud, taur, Vsyn,
                    trials, runs_per_trial
                )
                for mpf in hz_mpf_list
            ]
            for fut in as_completed(futs):
                parts.append(fut.result())
        return _merge_run_100hz_parts(hz_mpf_list, parts)

    except Exception as e:
        print("[parallel warning] ThreadPool fallback failed; running serial.")
        print("Reason:", repr(e))
        print(traceback.format_exc())

    # ---- Final fallback: single serial call (full list at once) ----
    return run_100hz(
        amp_100hz, gL, gk, gna, gel, gsyn,
        hz_mpf_list, noise_level, seed,
        max_L, k, x0, min_L, taud, taur, Vsyn,
        trials, runs_per_trial
    )




# =====================================================================================
# Hook into your existing main()
# =====================================================================================

def main():
    amplitude = 500
    gL, gk, gna, gel, gsyn = 0.03, 10, 300, 3.6, 0.08

    latency_mpf = [10]                 # latency sweep
    hz_mpf      = [0, 2, 4, 6, 8, 10]  # 100 Hz sweep

    noise_level = 10
    seed = None

    max_L, k, x0, min_L = 135, 0.129, 41, 34.5
    taud, taur, Vsyn = 3, 0.08, 0

    stimulus_mode   = 'both'    # 'latency', '100hz', or 'both'
    trials          = 1       # 1 = fast preview; 10 = better averages, matches plot in figure (+/- a few for noise randomness)
    runs_per_trial  = 10

    # equalize charge per pulse
    PULSE_LAT = 0.10   # ms
    PULSE_100 = 0.03   # ms
    amp_100hz = amplitude * (PULSE_LAT / PULSE_100)

    start_time = datetime.now()
    outputs = {}

    # ---------------- 100 Hz ----------------
    if stimulus_mode in ('100hz', 'both'):
        # PARALLELIZED sweep across mpf values
        out100 = run_100hz_parallel_over_mpf(
            amp_100hz, gL, gk, gna, gel, gsyn,
            hz_mpf, noise_level, seed,
            max_L, k, x0, min_L, taud, taur, Vsyn,
            trials, runs_per_trial,
            n_jobs=None  # or an explicit int like 4–8
        )


        # Your existing 100 Hz plots (unchanged)
        red_mpf, orange_mpf = hz_mpf[0], hz_mpf[1]
        plot_100hz_two_colors(
            out100['time'],
            out100['c0'],
            out100['c4'][red_mpf],
            out100['c4'][orange_mpf],
            f'{red_mpf} %', f'{orange_mpf} %'
        )
        plot_100hz_response_curve(out100['results_df'], runs_per_trial)

        # Single figure with BOTH sim curves + spreadsheet genotypes
        plot_100hz_combined(
            out100,
            runs_per_trial=runs_per_trial,
            mpf_to_plot=(10, 8, 6, 4, 0),  # adjust if your sweep differs
            excel_path=EXCEL_PATH,
            sheet_name=SHEET_NAME,
            savepath="100hz_sim_plus_genotypes.pdf"
        )

        outputs['100hz'] = out100

    # ---------------- latency ----------------
    if stimulus_mode in ('latency', 'both'):
        outlat = run_latency(
            amplitude, gL, gk, gna, gel, gsyn,
            latency_mpf, noise_level, seed,
            max_L, k, x0, min_L, taud, taur, Vsyn
        )

        plot_stimulus(outlat['time'], outlat['stimulus'], 'Latency')
        plot_latency_full(outlat['time'], outlat['v'],
                          outlat['idx0'], outlat['idx3'], outlat['latency'],
                          savepath='latency_full.pdf')

        post_dict = {
            f'{latency_mpf[0]} %': outlat['v'][3],
            '5 %': outlat['extra_v'][3]
        }
        plot_latency_panel(outlat['time'], outlat['v'][0], post_dict,
                           thresh=60, ylims=(-72, 80),
                           savepath='latency_small.pdf')

        outputs['latency'] = outlat

    print(f"Total run time: {datetime.now() - start_time}")
    return outputs


if __name__ == "__main__":
    results = main()


/var/folders/bc/364csh9x7ts0qcsvtp63gw91dhcrjl/T/ipykernel_62693/2468205018.py:460: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


[parallel warning] joblib 'loky' failed; falling back to ThreadPool.
Reason: KeyError('mpf')
Traceback (most recent call last):
  File "/var/folders/bc/364csh9x7ts0qcsvtp63gw91dhcrjl/T/ipykernel_62693/2460089887.py", line 237, in run_100hz_parallel_over_mpf
    return _merge_run_100hz_parts(hz_mpf_list, parts)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/bc/364csh9x7ts0qcsvtp63gw91dhcrjl/T/ipykernel_62693/2460089887.py", line 206, in _merge_run_100hz_parts
    merged["results_df"] = merged["results_df"].sort_values(["mpf", "interval"]).reset_index(drop=True)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/pandas/core/frame.py", line 7179, in sort_values
    keys = [self._get_label_or_level_values(x, axis=axis) for x in by]
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/pandas/core/generic.py", line 1911, in _ge

/var/folders/bc/364csh9x7ts0qcsvtp63gw91dhcrjl/T/ipykernel_62693/2468205018.py:460: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, pd.DataFrame(rows)], ignore_index=True)


[parallel warning] ThreadPool fallback failed; running serial.
Reason: KeyError('mpf')
Traceback (most recent call last):
  File "/var/folders/bc/364csh9x7ts0qcsvtp63gw91dhcrjl/T/ipykernel_62693/2460089887.py", line 260, in run_100hz_parallel_over_mpf
    return _merge_run_100hz_parts(hz_mpf_list, parts)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/bc/364csh9x7ts0qcsvtp63gw91dhcrjl/T/ipykernel_62693/2460089887.py", line 206, in _merge_run_100hz_parts
    merged["results_df"] = merged["results_df"].sort_values(["mpf", "interval"]).reset_index(drop=True)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/pandas/core/frame.py", line 7179, in sort_values
    keys = [self._get_label_or_level_values(x, axis=axis) for x in by]
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/pandas/core/generic.py", line 1911, in _get_labe

/var/folders/bc/364csh9x7ts0qcsvtp63gw91dhcrjl/T/ipykernel_62693/2468205018.py:460: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, pd.DataFrame(rows)], ignore_index=True)


# Sigmoid ggap

In [ ]:
def sigmoidal_ggap(model_percentage_filled, max_L=135, k=0.7, x0=6, min_L=34.5):
    if model_percentage_filled < 10.2:
        ggap_base = min_L + (max_L - min_L) / (1 + np.exp(-k * (model_percentage_filled - x0)))
    else:
        ggap_base = max_L
    return ggap_base

# Define a range of model_percentage_filled values
model_percentage_filled = np.linspace(0, 10.1, 300)

# Define different values of k to explore
k_values = [ 1,]

plt.figure(figsize=(10, 6))

# Loop over different values of k and plot each one
for k in k_values:
    sigmoid_conductance_values = [sigmoidal_ggap(pf, k=k) for pf in model_percentage_filled]
    plt.plot(model_percentage_filled, sigmoid_conductance_values, label=f'k={k}')

# Add labels, title, legend, and grid
plt.xlabel('Percentage Filled (%)')
plt.ylabel('Gap Junction Conductance (mS)')
plt.title('Modulation of Gap Junction Conductance via Sigmoid Function for Different k Values')
plt.axhline(y=135, color='r', linestyle='--', label='Max Conductance (max_L)')
plt.axhline(y=34.5, color='b', linestyle='--', label='Min Conductance (min_L)')
plt.legend(loc='upper left')
plt.grid(True)

# Show the plot
# plt.show()